In [11]:
import os
from dotenv import load_dotenv

load_dotenv(".env")

os.environ["KAGGLE_USERNAME"] = os.getenv("KAGGLE_USERNAME", "")
os.environ["KAGGLE_KEY"]      = os.getenv("KAGGLE_KEY", "")

DATA_RAW = os.path.join("..", "data", "raw")

print("Config loaded from .env")
print(f"KAGGLE_USERNAME : {os.getenv('KAGGLE_USERNAME')}")
print(f"KAGGLE_KEY      : {os.getenv('KAGGLE_KEY')[:6]}... (hidden)")

Config loaded from .env
KAGGLE_USERNAME : 
KAGGLE_KEY      : ... (hidden)


In [12]:
KAGGLE_DATASET = "gauravmalik26/food-delivery-dataset"
DATA_RAW       = os.path.join("..", "data", "raw")

os.makedirs(DATA_RAW, exist_ok=True)

print(f"Downloading: {KAGGLE_DATASET}")
os.system(f'kaggle datasets download -d {KAGGLE_DATASET} -p "{DATA_RAW}" --unzip')
print("Done!")

Downloading: gauravmalik26/food-delivery-dataset
Dataset URL: https://www.kaggle.com/datasets/gauravmalik26/food-delivery-dataset
License(s): other


100%|██████████| 1.95M/1.95M [00:01<00:00, 1.52MB/s]



Done!


In [13]:
# Verify Downloaded Files

files = os.listdir(DATA_RAW)
print(f"Files in {DATA_RAW}:")
for f in files:
    size = os.path.getsize(os.path.join(DATA_RAW, f))
    print(f"  {f:<35} {size/1024:>8.1f} KB")

Files in ../data/raw:
  test.csv                              1796.3 KB
  train.csv                             7590.1 KB
  Sample_Submission.csv                  267.1 KB


In [14]:
# Preview Data

import pandas as pd

TRAIN_CSV = os.path.join(DATA_RAW, "train.csv")
TEST_CSV  = os.path.join(DATA_RAW, "test.csv")

df_train = pd.read_csv(TRAIN_CSV)
df_test  = pd.read_csv(TEST_CSV)

print(f"train.csv : {df_train.shape[0]:,} rows × {df_train.shape[1]} columns")
print(f"test.csv  : {df_test.shape[0]:,} rows × {df_test.shape[1]} columns")
df_train.head(3)

train.csv : 45,593 rows × 20 columns
test.csv  : 11,399 rows × 19 columns


,ID,Delivery_person_ID,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Order_Date,Time_Orderd,Time_Order_picked,Weatherconditions,Road_traffic_density,Vehicle_condition,Type_of_order,Type_of_vehicle,multiple_deliveries,Festival,City,Time_taken(min)
0,0x4607,INDORES13DEL02,37,4.9,22.745049,75.892471,22.765049,75.912471,19-03-2022,11:30:00,11:45:00,conditions Sunny,High,2,Snack,motorcycle,0,No,Urban,(min) 24
1,0xb379,BANGRES18DEL02,34,4.5,12.913041,77.683237,13.043041,77.813237,25-03-2022,19:45:00,19:50:00,conditions Stormy,Jam,2,Snack,scooter,1,No,Metropolitian,(min) 33
2,0x5d6d,BANGRES19DEL01,23,4.4,12.914264,77.678400,12.924264,77.688400,19-03-2022,08:30:00,08:45:00,conditions Sandstorms,Low,0,Drinks,motorcycle,1,No,Urban,(min) 26


In [15]:
import pandas as pd
import os

DATA_RAW  = os.path.join("..", "data", "raw")
TRAIN_CSV = os.path.join(DATA_RAW, "train.csv")

df = pd.read_csv(TRAIN_CSV)
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print("\nColumns:")
for c in df.columns:
    sample = df[c].dropna().iloc[0]
    print(f"  {c:<45} sample: {sample}")

Shape: 45,593 rows × 20 columns

Columns:
  ID                                            sample: 0x4607 
  Delivery_person_ID                            sample: INDORES13DEL02 
  Delivery_person_Age                           sample: 37
  Delivery_person_Ratings                       sample: 4.9
  Restaurant_latitude                           sample: 22.745049
  Restaurant_longitude                          sample: 75.892471
  Delivery_location_latitude                    sample: 22.765049
  Delivery_location_longitude                   sample: 75.912471
  Order_Date                                    sample: 19-03-2022
  Time_Orderd                                   sample: 11:30:00
  Time_Order_picked                             sample: 11:45:00
  Weatherconditions                             sample: conditions Sunny
  Road_traffic_density                          sample: High 
  Vehicle_condition                             sample: 2
  Type_of_order                                 s

In [16]:
# Column Overview

print("Columns:")
for c in df_train.columns:
    dtype  = df_train[c].dtype
    nulls  = df_train[c].isnull().sum()
    sample = df_train[c].dropna().iloc[0] if not df_train[c].dropna().empty else "N/A"
    print(f"  {c:<40} dtype={str(dtype):<10} nulls={nulls:<6} sample={sample}")

Columns:
  ID                                       dtype=object     nulls=0      sample=0x4607 
  Delivery_person_ID                       dtype=object     nulls=0      sample=INDORES13DEL02 
  Delivery_person_Age                      dtype=object     nulls=0      sample=37
  Delivery_person_Ratings                  dtype=object     nulls=0      sample=4.9
  Restaurant_latitude                      dtype=float64    nulls=0      sample=22.745049
  Restaurant_longitude                     dtype=float64    nulls=0      sample=75.892471
  Delivery_location_latitude               dtype=float64    nulls=0      sample=22.765049
  Delivery_location_longitude              dtype=float64    nulls=0      sample=75.912471
  Order_Date                               dtype=object     nulls=0      sample=19-03-2022
  Time_Orderd                              dtype=object     nulls=0      sample=11:30:00
  Time_Order_picked                        dtype=object     nulls=0      sample=11:45:00
  Weatherco

In [17]:
# Null Summary

null_summary = df_train.isnull().sum().reset_index()
null_summary.columns = ["column", "null_count"]
null_summary["null_%"] = (null_summary["null_count"] / len(df_train) * 100).round(2)
null_summary[null_summary["null_count"] > 0].sort_values("null_%", ascending=False)

,column,null_count,null_%
